# Tukey API

- This notebook tours the Tukey API: a Gym-style environment that hides a
  random causal graph, statistical tools an agent can call on the training
  data, a verifiable score, and a policy trained to choose the tools
- The pedagogical arc:
  - Part 1: check the environment and the pinned scientific stack
  - Part 2: sample a hidden graph and score candidate graphs against it
  - Part 3: call the statistical tools and run the scripted agent
  - Part 4: run a trained policy and read its tool-call trace

In [1]:
%load_ext autoreload
%autoreload 2

import logging

In [2]:
import helpers.hintrospection as hintros
import helpers.hnotebook as hnotebook
import tukey_utils as utils

# Initialize notebook configuration and logging.
hnotebook.config_notebook()
_LOG = logging.getLogger(__name__)
utils.init_loggers(_LOG)

# Convert `display` into `print()` when running outside IPython.
try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore

INFO  > /usr/local/lib/python3.12/site-packages/ipykernel_launcher.py -f /tmp/tmpkh6wi99k.json --HistoryManager.hist_file=:memory:


## Library Overview

- **What problem it solves**: learning which statistical test to run next on
  an unknown dataset, rewarded by how well the discovered graph matches the
  hidden graph that generated the data
- **Key abstraction**: an episode = sample a hidden graph and data, call
  tools on the training rows, emit a graph, receive a verifiable score
- **Mental model**:

| Object | Description | Comments |
|--------|-------------|----------|
| `GraphConfig` | Generator settings | Family, size, noise, scaling |
| `TukeyEnv.reset()` | Samples a hidden graph and data | The graph stays private |
| `TukeyEnv.step()` | Runs one tool call or emits a graph | Emitting ends the episode |
| `score_graph()` | Verifiable score | Structure plus held-out prediction |

- **Key classes**:
  - `TukeyEnv`: the environment
  - `TukeyPolicy`: the trained tool-choosing policy

# Part 1: Environment check

## Cell 1.1: Pinned scientific stack

**Goal**
- Confirm the container runs the pinned versions the results were produced
  with

In [3]:
# Report the version of every package the project depends on.
versions = utils.cell1_1_get_package_versions()
display(versions)
# Outcome: one row per package, matching `requirements.txt`.

,package,version
0,numpy,2.5.4
1,pandas,3.0.6
2,scipy,1.18.1
3,scikit-learn,1.9.1
4,networkx,3.7
5,causal-learn,0.1.4.8
6,lingam,1.12.2
7,CausalDisco,0.2.4
8,pgmpy,1.1.2
9,torch,2.14.0+cpu


**Implementation** `cell1_1_get_package_versions()`
- Reads each version from the installed distribution metadata

In [4]:
hintros.print_obj_info(utils.cell1_1_get_package_versions)

- `cell1_1_get_package_versions(*, packages: Tuple[str, ...] = ('numpy', 'pandas', 'scipy', 'scikit-learn', 'networkx', 'causal-learn', 'lingam', 'CausalDisco', 'pgmpy', 'torch')) -> pandas.DataFrame`  
  Report the installed version of each package the project depends on.

https://github.com/Rayhanpatel/gpsaggese.github.io/blob/UmdTask628_MSML610_Fall2026_Tukey_RL_agent_for_automated_EDA/class_project/msml610/Fall2026/projects/UmdTask628_MSML610_Fall2026_Tukey_RL_agent_for_automated_EDA/tukey_utils.py#L52
